# 🧬 QML TB Patient Analyzer — 8-Qubit Production Pipeline

**Architecture:** 8 Qubits | 16 Features | 4 Layers | 256-dim Hilbert Space | 3M Patients

### Instructions:
1. **Settings → Accelerator → None** (PennyLane quantum simulator runs on CPU)
2. **Run All**
3. Download `tb_weights.npy` and `classical_baseline_results.json` when done

In [ ]:
!pip install -q pennylane scikit-learn

In [ ]:
import os, time, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pennylane as qml
print(f'PyTorch {torch.__version__} | PennyLane {qml.__version__}')
print('NOTE: PennyLane default.qubit is a CPU quantum simulator — no GPU needed!')

---
## 1. Dataset Generation — 3 Million TB Patients (16 Features)


In [ ]:
FEATURE_NAMES = [
    'heart_rate', 'spo2', 'resp_rate', 'temperature',
    'wbc_count', 'esr', 'crp', 'lymphocyte_pct',
    'hemoglobin', 'albumin',
    'xray_opacity', 'xray_cavity', 'xray_nodule', 'xray_pleural',
    'ada_level', 'mantoux_mm'
]
N_FEATURES = 16

MEANS = np.array([78.0, 97.5, 16.0, 36.8, 7.0, 10.0, 2.0, 30.0, 13.5, 4.0, 0.08, 0.04, 0.06, 0.05, 15.0, 5.0])
STDS  = np.array([8.0, 0.8, 2.0, 0.3, 1.5, 5.0, 1.0, 5.0, 1.2, 0.4, 0.04, 0.025, 0.03, 0.025, 8.0, 4.0])
LOWS  = np.array([40, 70, 8, 35.0, 2.0, 0, 0, 5, 5.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0, 0])
HIGHS = np.array([180, 100, 40, 42.0, 30.0, 120, 200, 60, 20.0, 5.5, 1.0, 1.0, 1.0, 1.0, 150, 30])
DRIFT = np.array([+1,-1,+1,+1, +1,+1,+1,+1, -1,-1, +1,+1,+1,+1, +1,+1], dtype=np.float32)
TIER_MAG = {1:(0.20,0.45), 2:(0.45,0.80), 3:(0.80,1.30), 4:(1.30,1.80), 5:(1.80,2.50), 6:(2.50,4.00)}

def build_corr_matrix():
    C = np.eye(N_FEATURES, dtype=np.float64)
    pairs = [
        (0,2,0.35),(0,3,0.42),(1,2,-0.32),(1,3,-0.18),(2,3,0.25),
        (4,5,0.40),(5,6,0.62),(4,6,0.38),(4,7,0.30),(8,9,0.35),
        (5,8,-0.28),(6,9,-0.25),(3,5,0.32),(3,6,0.35),(0,8,-0.15),
        (1,8,0.20),(10,11,0.52),(10,12,0.48),(10,13,0.30),(11,12,0.28),
        (13,14,0.45),(1,10,-0.28),(2,10,0.22),(3,10,0.18),(5,10,0.38),
        (6,11,0.32),(8,10,-0.22),(9,10,-0.20),(14,5,0.40),(14,6,0.35),
        (14,7,0.42),(14,10,0.30),(15,3,0.20),(15,7,0.35),(15,14,0.50),(15,10,0.18),
    ]
    for i,j,v in pairs:
        C[i,j] = C[j,i] = v
    eigvals = np.linalg.eigvalsh(C)
    if np.min(eigvals) < 0:
        C += (-np.min(eigvals) + 0.02) * np.eye(N_FEATURES)
        d = np.sqrt(np.diag(C))
        C = C / np.outer(d, d)
    return C

print('Feature specs loaded. 16 features, 7 tiers.')

In [ ]:
def generate_dataset(n_total=3_000_000):
    print(f'Generating {n_total:,} TB patients...')
    t0 = time.time()
    corr = build_corr_matrix()
    cov = np.outer(STDS, STDS) * corr
    tier_dist = {0:0.30, 1:0.15, 2:0.15, 3:0.15, 4:0.10, 5:0.10, 6:0.05}
    counts = {t: int(n_total*f) for t,f in tier_dist.items()}
    counts[0] += n_total - sum(counts.values())
    
    all_X, all_y, all_t = [], [], []
    for tier, count in counts.items():
        samples = np.random.multivariate_normal(MEANS, cov, size=count).astype(np.float32)
        ages = np.concatenate([np.random.normal(30,8,count//2), np.random.normal(58,10,count-count//2)])
        ages = np.clip(ages, 18, 90); np.random.shuffle(ages)
        age_factor = (ages - 45) / 30
        samples[:, 0] += age_factor * 3.0
        samples[:, 5] += age_factor * 4.0
        samples[:, 8] -= age_factor * 0.5
        is_male = np.random.random(count) < 0.60
        samples[is_male, 8] += 1.0
        samples[~is_male, 8] -= 0.5
        dm = np.random.random(count) < 0.15
        samples[dm, 6] += np.random.uniform(0.5, 2.0, dm.sum())
        samples[dm, 5] += np.random.uniform(2.0, 5.0, dm.sum())
        hiv = np.random.random(count) < 0.05
        samples[hiv, 7] -= np.random.uniform(5.0, 15.0, hiv.sum())
        samples[hiv, 15] -= np.random.uniform(3.0, 6.0, hiv.sum())
        if tier > 0:
            lo, hi = TIER_MAG[tier]
            mag = np.random.uniform(lo, hi, (count, 1))
            noise = 1.0 + np.random.normal(0, 0.20, (count, N_FEATURES))
            leader = (np.random.random((count, N_FEATURES)) > 0.3).astype(np.float32)
            samples += DRIFT * STDS * mag * noise * leader
        for i in range(N_FEATURES):
            samples[:, i] = np.clip(samples[:, i], LOWS[i], HIGHS[i])
        all_X.append(samples)
        all_y.append(np.ones(count, dtype=np.float32) if tier > 0 else np.zeros(count, dtype=np.float32))
        all_t.append(np.full(count, tier, dtype=np.int32))
    
    X = np.concatenate(all_X); y = np.concatenate(all_y); t = np.concatenate(all_t)
    idx = np.arange(len(X)); np.random.shuffle(idx)
    X, y, t = X[idx], y[idx], t[idx]
    os.makedirs('data', exist_ok=True)
    np.save('data/tb_features.npy', X)
    np.save('data/tb_labels.npy', y)
    np.save('data/tb_tiers.npy', t)
    print(f'Done in {time.time()-t0:.1f}s | Shape: {X.shape}')
    for tid in range(7):
        label = 'Healthy' if tid == 0 else f'Tier-{tid}'
        print(f'  {label:10s}: {(t==tid).sum():>10,}')
    return X, y, t

X_all, y_all, tiers_all = generate_dataset(3_000_000)

---
## 2. Classical ML Baseline (4 Models)


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

n_cl = 300_000
idx_cl = np.random.choice(len(X_all), n_cl, replace=False)
X_cl = StandardScaler().fit_transform(X_all[idx_cl])
y_cl = y_all[idx_cl]; t_cl = tiers_all[idx_cl]
X_tr, X_te, y_tr, y_te, t_tr, t_te = train_test_split(X_cl, y_cl, t_cl, test_size=0.2, stratify=y_cl)

classical_results = {}
models = {
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'MLP Neural Net': MLPClassifier(hidden_layer_sizes=(128,64,32), max_iter=300, random_state=42),
}

for name, model in models.items():
    print(f'Training {name}...')
    t0 = time.time()
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:,1]
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_prob)
    tier_accs = {}
    for tid in range(7):
        mask = t_te == tid
        if mask.sum() > 0:
            tier_accs[f'tier_{tid}'] = round(accuracy_score(y_te[mask], y_pred[mask])*100, 2)
    classical_results[name] = {'accuracy': round(acc*100,2), 'f1': round(f1*100,2), 'auc': round(auc*100,2), 'tiers': tier_accs}
    print(f'  Acc: {acc*100:.2f}% | F1: {f1*100:.2f}% | AUC: {auc*100:.2f}% | Time: {time.time()-t0:.1f}s')
    print(f'  Tier Accuracy: {tier_accs}')

os.makedirs('saved_model', exist_ok=True)
with open('saved_model/classical_baseline_results.json', 'w') as f:
    json.dump(classical_results, f, indent=2)
print('\nClassical baseline saved!')

---
## 3. Quantum Model — 8 Qubits, 4 Layers, 256-dim Hilbert Space

**All on CPU.** PennyLane `default.qubit` is a state-vector simulator that runs matrix ops on CPU.
We use a small 4,000 patient subset to keep training fast (~5-10 min).

In [ ]:
N_QUBITS = 8
N_LAYERS = 4

dev = qml.device('default.qubit', wires=N_QUBITS)

NORM_BASES  = torch.tensor([78.0, 97.5, 16.0, 36.8, 7.0, 10.0, 2.0, 30.0, 13.5, 4.0, 0.08, 0.04, 0.06, 0.05, 15.0, 5.0])
NORM_SCALES = torch.tensor([16.0, -5.0, 8.0, 1.5, 6.0, 20.0, 8.0, 15.0, -4.0, -1.5, 0.30, 0.25, 0.25, 0.20, 40.0, 10.0])

def normalize_features(X_raw):
    return torch.clamp(((X_raw - NORM_BASES) / NORM_SCALES) * math.pi, -math.pi, math.pi)

@qml.qnode(dev, interface='torch', diff_method='backprop')
def quantum_circuit(inputs, weights):
    # Layer 0: Vitals (features 0-7)
    for q in range(N_QUBITS):
        qml.RY(inputs[q], wires=q)
    for q in range(N_QUBITS):
        qml.Rot(weights[0,q,0], weights[0,q,1], weights[0,q,2], wires=q)
    for i in range(N_QUBITS): qml.CNOT(wires=[i,(i+1)%N_QUBITS])
    for i in range(0,N_QUBITS-1,2): qml.CNOT(wires=[i,i+1])
    for i in range(4): qml.CNOT(wires=[i,i+4])

    # Layer 1: Blood/Tests (features 8-15)
    for q in range(N_QUBITS):
        qml.RY(inputs[q+8], wires=q)
    for q in range(N_QUBITS):
        qml.Rot(weights[1,q,0], weights[1,q,1], weights[1,q,2], wires=q)
    for i in range(N_QUBITS): qml.CNOT(wires=[i,(i+1)%N_QUBITS])
    for i in range(0,N_QUBITS-1,2): qml.CNOT(wires=[i,i+1])
    for i in range(4): qml.CNOT(wires=[i,i+4])

    # Layer 2: Full re-upload (RY=vitals, RZ=blood)
    for q in range(N_QUBITS):
        qml.RY(inputs[q], wires=q)
        qml.RZ(inputs[q+8], wires=q)
    for q in range(N_QUBITS):
        qml.Rot(weights[2,q,0], weights[2,q,1], weights[2,q,2], wires=q)
    for i in range(N_QUBITS): qml.CNOT(wires=[i,(i+1)%N_QUBITS])
    for i in range(0,N_QUBITS-1,2): qml.CNOT(wires=[i,i+1])
    for i in range(4): qml.CNOT(wires=[i,i+4])

    # Layer 3: Cross-domain mixing (vital x xray interleaved)
    cross = [0, 10, 1, 11, 2, 12, 3, 13]
    for q in range(N_QUBITS):
        qml.RY(inputs[cross[q]], wires=q)
    for q in range(4):
        qml.RZ(inputs[14], wires=q)
        qml.RZ(inputs[15], wires=q+4)
    for q in range(N_QUBITS):
        qml.Rot(weights[3,q,0], weights[3,q,1], weights[3,q,2], wires=q)
    for i in range(N_QUBITS): qml.CNOT(wires=[i,(i+1)%N_QUBITS])
    for i in range(0,N_QUBITS-1,2): qml.CNOT(wires=[i,i+1])
    for i in range(4): qml.CNOT(wires=[i,i+4])

    return qml.expval(sum(qml.PauliZ(i) for i in range(N_QUBITS)))

class QMLTBPredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(N_LAYERS, N_QUBITS, 3) * 0.3)
    
    def forward(self, x):
        # Sequential per-sample execution (reliable, no batching issues)
        results = [quantum_circuit(x[i], self.weights) for i in range(x.shape[0])]
        return torch.stack(results)

print(f'Circuit: {N_QUBITS} qubits, {N_LAYERS} layers, {N_LAYERS*N_QUBITS*3} quantum params')
print(f'Hilbert Space: {2**N_QUBITS} dimensions')

In [ ]:
# ── TRAINING ──
TRAIN_SUBSET = 4_000
BATCH_SIZE = 16
EPOCHS = 20
LR = 0.05

X_raw_np = np.load('data/tb_features.npy')
y_raw_np = np.load('data/tb_labels.npy')

idx = np.random.choice(len(X_raw_np), TRAIN_SUBSET, replace=False)
X_sub = torch.tensor(X_raw_np[idx], dtype=torch.float32)
y_sub = torch.tensor(y_raw_np[idx], dtype=torch.float32)
X_norm = normalize_features(X_sub)

n_val = int(TRAIN_SUBSET * 0.20)
n_train = TRAIN_SUBSET - n_val
ds = TensorDataset(X_norm, y_sub)
train_ds, val_ds = torch.utils.data.random_split(ds, [n_train, n_val])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, drop_last=True)

model = QMLTBPredictor()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

print(f'Train: {n_train:,} | Val: {n_val:,} | Batch: {BATCH_SIZE}')
print(f'Batches per epoch: {len(train_loader)} | Expected ~2-4 min per epoch')

best_vl = float('inf')
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    tl, tc, tt = 0., 0, 0
    for bi, (bx, by) in enumerate(train_loader):
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        tl += loss.item() * bx.size(0)
        tc += ((torch.sigmoid(out) >= 0.5).float() == by).sum().item()
        tt += bx.size(0)
        if (bi + 1) % 50 == 0:
            print(f'  batch {bi+1}/{len(train_loader)} | loss {loss.item():.4f}')
    tl /= tt; ta = 100 * tc / tt
    
    model.eval()
    vl, vc, vt = 0., 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            out = model(bx)
            loss = criterion(out, by)
            vl += loss.item() * bx.size(0)
            vc += ((torch.sigmoid(out) >= 0.5).float() == by).sum().item()
            vt += bx.size(0)
    vl /= max(vt, 1); va = 100 * vc / max(vt, 1)
    scheduler.step()
    
    elapsed = time.time() - t0
    print(f'Ep {epoch+1:>3}/{EPOCHS} | TrLoss {tl:.4f} | TrAcc {ta:.1f}% | VaLoss {vl:.4f} | VaAcc {va:.1f}% | {elapsed:.1f}s')
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta); history['val_acc'].append(va)
    
    if vl < best_vl:
        best_vl = vl
        np.save('saved_model/tb_weights.npy', model.weights.detach().cpu().numpy())
        torch.save(model.state_dict(), 'saved_model/tb_full_model.pt')
        print(f'  ✓ Saved best model (val_loss={vl:.4f})')

with open('saved_model/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\n✓ Training Complete! Best Val Loss: {best_vl:.4f}')
print(f'✓ Download: saved_model/tb_weights.npy')

---
## 4. Results Summary


In [ ]:
print('\n' + '='*60)
print('  RESULTS: Classical ML vs Quantum ML')
print('='*60)
print('\n--- Classical ML ---')
for name, r in classical_results.items():
    print(f'  {name:25s} | Acc: {r["accuracy"]:5.2f}% | Tier-1: {r["tiers"].get("tier_1","N/A")}%')
print('\n--- Quantum ML ---')
print(f'  8-Qubit QML (256-dim)    | Val Acc: {history["val_acc"][-1]:.2f}%')
print(f'  Best Val Loss:           | {best_vl:.4f}')
print('='*60)

In [ ]:
from IPython.display import FileLink
display(FileLink('saved_model/tb_weights.npy'))
display(FileLink('saved_model/classical_baseline_results.json'))
display(FileLink('saved_model/training_history.json'))